# DualAMP-Lite · Colab 运行（T4 16 GB）

免费 Colab 的会话随时会断，所以这个 Notebook 的设计原则是**每步都能续跑**：
特征 npz 与模型结果都落在 Google Drive 上，重连后继续跑同一批 `--seeds` 即可。

**建议：**
1. 运行时 → 更改运行时类型 → **T4 GPU**；
2. 把原始数据放到 Drive：`MyDrive/DualAMP-Lite/raw/`（含 `sources.yaml`）；
3. 首次跑会安装 `fair-esm`（约 1 分钟）并下载 140 MB 的 ESM-2 权重到 Drive 缓存。

没有 `raw/sources.yaml` 时自动降级为合成 demo（只验证流水线，指标不可引用）。

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:  # not on Colab
    print("Drive 未挂载（非 Colab 环境？）:", exc)

In [ ]:
import importlib.util, os, shutil, subprocess, sys, textwrap
from pathlib import Path

RANK = None
def pip(*args: str) -> None:
    """Install without letting pip replace the image's CUDA build of torch."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

def run(args: list[str], check: bool = True) -> int:
    cmd = [sys.executable, "-m", "dualamp_lite", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=CODE, check=False)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc.returncode

def human(n: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} GB"

USE_DRIVE = os.environ.get("DUALAMP_DRIVE", "1") == "1"
DRIVE_ROOT = Path(os.environ.get("DUALAMP_DRIVE_ROOT", "/content/drive/MyDrive/DualAMP-Lite"))
REPO    = Path(os.environ.get("DUALAMP_REPO", "/content/leanrea"))
WORK    = Path(os.environ.get("DUALAMP_WORKDIR", str(DRIVE_ROOT / "work")))
RAW     = Path(os.environ.get("DUALAMP_RAW", str(DRIVE_ROOT / "raw")))
WEIGHTS = Path(os.environ.get("DUALAMP_WEIGHTS", str(DRIVE_ROOT / "esm2")))
BRANCH  = os.environ.get("DUALAMP_BRANCH", "arena/4ecb70f1-leanrea")
REPO_URL = "https://github.com/hukeying85-ux/leanrea"

In [ ]:
for module, package in (("Bio", "biopython"), ("yaml", "pyyaml"), ("joblib", "joblib")):
    if importlib.util.find_spec(module) is None:
        pip(package)
if importlib.util.find_spec("esm") is None:
    pip("fair-esm", "--no-deps")

import numpy, pandas, sklearn, scipy, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), human(torch.cuda.get_device_properties(0).total_memory))

# ESM-2 权重缓存到 Drive，避免每次会话重新下载
os.environ.setdefault("TORCH_HOME", str(WORK.parent / "cache" / "torch"))
WORK.mkdir(parents=True, exist_ok=True)
print("workdir:", WORK)

In [ ]:
if not (REPO / "dualamp-lite" / "dualamp_lite").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=False)
CODE = REPO / "dualamp-lite"
assert (CODE / "dualamp_lite" / "cli.py").exists(), (
    "代码拉取失败：检查网络，或手动上传 dualamp-lite 目录到 /content/leanrea/dualamp-lite"
)
run(["--workdir", str(WORK), "check"])

In [ ]:
manifest = RAW / "sources.yaml"
DEMO = not manifest.exists()
SOURCE_ARGS = ["--demo"] if DEMO else ["--manifest", str(manifest)]
print("=" * 78)
print("⚠️  未找到 %s -> 合成 demo 模式（指标不可引用）" % manifest if DEMO
      else "✅ 真实数据: %s" % manifest)
print("=" * 78)

# Colab 上把大文件放本地磁盘更快，最后再同步回 Drive
LOCAL_CACHE = Path("/content/dualamp_cache")
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- step 1：数据集（D2/D3）--------------------------------------------------
run(["build", *SOURCE_ARGS, "--workdir", str(WORK), "--cluster-method", "kmer" if DEMO else "auto"])

import json
print(json.dumps(json.loads((WORK / "dataset" / "dataset_summary.json").read_text()),
                 ensure_ascii=False, indent=2)[:2000])

In [ ]:
# --- step 2：特征（D4）-------------------------------------------------------
feat_cmd = ["features", "--workdir", str(WORK), "--batch-size", "64"]
if DEMO:
    feat_cmd += ["--backend", "mock"]
elif WEIGHTS.is_dir() and list(WEIGHTS.glob("*.pt")):
    feat_cmd += ["--backend", "fair-esm", "--weights-dir", str(WEIGHTS), "--local-files-only"]
run(feat_cmd)

## 训练：按批次跑，断了续跑

下面的 `SEEDS` 决定“这一次训练算哪一批”。默认 1–3，重连后再跑 4–5 的 cell 即可；
已经完成的模型（同 seeds）会自动跳过。

In [ ]:
SEEDS = os.environ.get("DUALAMP_SEEDS", "1,2")
SCALE = ["--seeds", SEEDS, "--folds", "3" if DEMO else "5",
         "--epochs", "8" if DEMO else "60", "--n-boot", "300" if DEMO else "2000"]
run(["train", "--workdir", str(WORK), "--models", "M0_RF,M0_HGB", *SCALE])
run(["train", "--workdir", str(WORK), "--models", "M1,M2", *SCALE])

In [ ]:
run(["train", "--workdir", str(WORK), "--models", "M3,M3_NOPHYS", *SCALE])

In [ ]:
# 第二批 seeds（可选）：把上面两个 cell 的 SEEDS 改成 "4,5" 后重跑即可
RUN_SECOND_BATCH = os.environ.get("DUALAMP_SECOND_BATCH", "0") == "1"
if RUN_SECOND_BATCH:
    SCALE2 = ["--seeds", "4,5", "--folds", "5", "--epochs", "60", "--n-boot", "2000"]
    run(["train", "--workdir", str(WORK), "--models", "M0_RF,M0_HGB,M1,M2,M3,M3_NOPHYS", *SCALE2])
else:
    print("跳过第二批 seeds（设 DUALAMP_SECOND_BATCH=1 启用）")

In [ ]:
# --- step 3：报告（D9–D11 + D15 指纹）---------------------------------------
run(["report", "--workdir", str(WORK)])

import pandas as pd
metrics = pd.read_csv(WORK / "manuscript" / "D10_test_metrics.csv")
cols = [c for c in ["model", "auroc", "auroc_ci_low", "auroc_ci_high", "auprc", "mcc",
                    "sensitivity", "specificity"] if c in metrics.columns]
display(metrics[cols].sort_values("auroc", ascending=False).round(3))

from IPython.display import Markdown
Markdown((WORK / "manuscript" / "D10_evaluation_report.md").read_text(encoding="utf-8")[:6000])

## 注意

* 免费 Colab 的 GPU 时段不保证，长时间训练请拆成多个 cell，并保留 `WORK` 在 Drive 上；
* Drive 读写较慢，`features` 步骤建议只做一次（缓存会复用）；
* 断线后重新运行：挂载 Drive → 装依赖 → `check` → 直接跳到没完成的训练 cell；
* 真实数据的 D11/D12/D13 与 Kaggle 版本一致，见 `README.md` 第 5 节与 `docs/PLAN_MAPPING.md`。